[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_58_Typer_CLI_PyPI_Packaging.ipynb)

# Lesson 58 — paper-distiller: Typer CLI + PyPI Packaging
**Phase 6 · OSS Builder · Lesson 3 of 10**

---

## Where we are

| Lesson | Topic | Status |
|--------|-------|--------|
| L56 | Phase 6 Kickoff — OSS project selection, scaffold, FetchLayer + ExtractLayer | ✅ Done |
| L57 | Section Detection, Scanned PDF Fallback, Batch Distiller, Golden Eval Harness | ✅ Done |
| **L58** | **Typer CLI + PyPI Packaging** | **← You are here** |
| L59 | FastAPI Web API — `/distill`, `/batch`, `/health` endpoints | 🔜 |
| L60 | External Data: Semantic Scholar + S2ORC integration | 🔜 |
| L61 | Safety & Guardrails — prompt injection defence, cost governance | 🔜 |
| L62 | CI/CD — GitHub Actions, codecov, nightly golden eval | 🔜 |
| L63 | OSS Growth — README, docs site, contributing guide | 🔜 |
| L64 | Agent integration — paper-distiller as an MCP tool | 🔜 |
| L65 | Launch Day — public GitHub release, PyPI v1.0 | 🔜 |

---

## What you'll build today

Right now `paper-distiller` is a Python library — you can `import` it, but you can't call it from a terminal. After this lesson:

```bash
# Install from PyPI
pip install paper-distiller

# Distill a single paper
paper-distiller distill 1706.03762

# Batch-distill from a file
paper-distiller batch arxiv_ids.txt --output digests.jsonl

# Run the golden eval harness
paper-distiller eval --threshold 0.75

# Show version
paper-distiller --version
```

**Concepts covered:**
- Typer — the modern Python CLI framework (Click under the hood, but type-annotated)
- Rich integration — progress bars, panels, tables in the terminal
- `pyproject.toml` entry points — how `paper-distiller` becomes a shell command
- Building a wheel + sdist with `python -m build`
- Publishing to TestPyPI and real PyPI
- OIDC Trusted Publishing — no API tokens needed in CI

---

## Why Typer, not argparse or Click?

| Framework | DX | Type safety | Auto docs | Best for |
|-----------|----|-----------|-----------|---------|
| `argparse` | Verbose boilerplate | None | Manual | stdlib fallback |
| `click` | Decorators, good DX | Manual `type=` | Auto `--help` | Battle-tested APIs |
| **`typer`** | **Python type hints = CLI** | **Full mypy** | **Auto + rich** | **Modern OSS tools** |
| `fire` | Zero config | None | Reflected | Quick scripts only |

**Typer's big idea:** your function signature IS your CLI schema.
```python
def distill(arxiv_id: str, model: str = "claude-sonnet-4-6", max_cost: float = 0.05):
    ...
```
Typer reads the annotations and generates `--arxiv-id`, `--model`, `--max-cost` flags, help text, type validation, and shell completion — all for free.

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
# Run this first. Takes ~60s in Colab.
!pip install typer[all] anthropic pdfplumber pydantic requests rich nest_asyncio pdf2image Pillow PyMuPDF -q

# poppler is needed for pdf2image
!apt-get install -y poppler-utils > /dev/null 2>&1

# Build tools for PyPI packaging
!pip install build twine -q

import os, nest_asyncio
nest_asyncio.apply()

# Load your Anthropic API key from Colab Secrets
# (Sidebar → 🔑 Secrets → Add ANTHROPIC_API_KEY)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("⚠️  Set ANTHROPIC_API_KEY manually: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")

print("Setup complete.")

## 1. Core Typer Concepts

Before building the full CLI, let's understand the three building blocks:

```
app = typer.Typer()          ← the application object
@app.command()               ← registers a subcommand
def distill(arxiv_id: str):  ← function signature = CLI flags
    ...
```

**Arguments vs Options:**
- `arxiv_id: str` → positional argument (required: `paper-distiller distill 1706.03762`)
- `model: str = "claude-sonnet-4-6"` → optional flag (`--model claude-haiku-4-5`)
- `typer.Argument(...)` → explicit positional with help text
- `typer.Option(...)` → explicit option with short flags, env var binding, etc.

**Annotations Typer understands:**
- `str`, `int`, `float`, `bool` → basic types with auto-validation
- `Path` → checks file existence automatically
- `Optional[str]` → makes the flag nullable
- `List[str]` → repeatable flag (`--tag ml --tag nlp`)
- `Enum` → restricts to allowed values, shows choices in `--help`

In [ ]:
# ── CELL 2: Typer 5-minute primer ──────────────────────────────────────────────
import typer
from typing import Optional, List
from pathlib import Path
from enum import Enum
from rich.console import Console
from rich.table import Table

console = Console()

# ---- Minimal app to learn the building blocks --------------------------------
demo_app = typer.Typer(name="demo", help="Typer primer demo")

class OutputFmt(str, Enum):
    json = "json"
    markdown = "markdown"
    rich = "rich"

@demo_app.command("greet")
def greet(
    name: str = typer.Argument(..., help="Your name"),
    count: int = typer.Option(1, "--count", "-n", help="How many times"),
    fmt: OutputFmt = typer.Option(OutputFmt.rich, help="Output format"),
    shout: bool = typer.Option(False, "--shout", "-s", help="UPPERCASE output"),
):
    """Greet NAME count times."""
    for i in range(count):
        msg = f"Hello, {name}! ({i+1}/{count})"
        if shout:
            msg = msg.upper()
        if fmt == OutputFmt.rich:
            console.print(f"[bold green]{msg}[/bold green]")
        elif fmt == OutputFmt.json:
            import json
            print(json.dumps({"greeting": msg, "index": i}))
        else:
            print(f"**{msg}**")

@demo_app.command("list-files")
def list_files(
    directory: Path = typer.Argument(Path("."), help="Directory to list"),
    pattern: str = typer.Option("*", help="Glob pattern"),
):
    """List files matching PATTERN in DIRECTORY."""
    files = sorted(directory.glob(pattern))
    table = Table("Name", "Size", "Type")
    for f in files[:10]:
        kind = "dir" if f.is_dir() else "file"
        size = f"{f.stat().st_size:,}" if f.is_file() else "-"
        table.add_row(f.name, size, kind)
    console.print(table)
    console.print(f"[dim]Showing {min(len(files),10)} of {len(files)} matches[/dim]")

# In Colab we call the command functions directly (no subprocess)
console.rule("[bold]greet demo — name='Gourav', count=3, shout=True")
greet(name="Gourav", count=3, fmt=OutputFmt.rich, shout=True)

console.rule("[bold]greet demo — fmt='json', count=2")
greet(name="World", count=2, fmt=OutputFmt.json, shout=False)

console.rule("[bold]list-files demo")
list_files(directory=Path("/content"), pattern="*")

## 2. The paper-distiller module files (recap)

Lessons 56–57 built four modules. We'll write them to disk here so the CLI can import them:

```
paper_distiller/
├── __init__.py
├── models.py       ← PaperDigest Pydantic model
├── fetch.py        ← fetch arXiv metadata + PDF bytes
├── extract.py      ← SectionDetector + ScannedPDFFallback + ExtractLayer
├── codegen.py      ← CodeLayer (Haiku code snippet)
├── pipeline.py     ← top-level distill() + batch_distill()
├── evals/
│   └── golden_harness.py  ← GOLDEN_SET + run_golden_eval()
└── cli.py          ← ← ← THIS IS WHAT WE BUILD TODAY
```

We'll write stub versions of the modules so the CLI wiring runs end-to-end in Colab.

In [ ]:
# ── CELL 3: Write paper_distiller module stubs ─────────────────────────────────
import os, textwrap

BASE = "/content/paper_distiller"
os.makedirs(f"{BASE}/evals", exist_ok=True)

# ---------- models.py ---------------------------------------------------------
with open(f"{BASE}/models.py", "w") as f:
    f.write(textwrap.dedent("""
        from pydantic import BaseModel
        from typing import Optional, List

        class PaperDigest(BaseModel):
            arxiv_id: str
            title: str
            one_liner: str
            method_summary: str
            key_results: str
            prerequisites: List[str] = []
            limitations: str = ""
            practitioner_tldr: str = ""
            tags: List[str] = []
            code_example: str = ""
            cost_usd: float = 0.0
            method: str = "native"
    """))

# ---------- fetch.py ----------------------------------------------------------
with open(f"{BASE}/fetch.py", "w") as f:
    f.write(textwrap.dedent("""
        import re, requests
        from typing import Optional

        def parse_arxiv_id(raw: str) -> str:
            """Accept bare ID, abs URL, or PDF URL; return bare ID like '1706.03762'."""
            m = re.search(r'(\\d{4}\\.\\d{4,5}(?:v\\d+)?)', raw)
            if m:
                return m.group(1)
            raise ValueError(f"Cannot parse arXiv ID from: {raw!r}")

        def fetch_metadata(arxiv_id: str) -> dict:
            url = f"https://export.arxiv.org/abs/{arxiv_id}"
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            title = re.search(r'<title>(.*?)</title>', r.text, re.S)
            return {"arxiv_id": arxiv_id, "title": (title.group(1).strip() if title else arxiv_id)}

        def fetch_pdf_bytes(arxiv_id: str) -> bytes:
            url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            return r.content
    """))

# ---------- extract.py --------------------------------------------------------
with open(f"{BASE}/extract.py", "w") as f:
    f.write(textwrap.dedent("""
        import os, anthropic, pdfplumber
        from io import BytesIO
        from .models import PaperDigest

        _client = None
        def _get_client():
            global _client
            if _client is None:
                _client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
            return _client

        EXTRACT_TOOL = {
            "name": "extract_digest",
            "description": "Extract a structured digest from a research paper.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "one_liner": {"type": "string"},
                    "method_summary": {"type": "string"},
                    "key_results": {"type": "string"},
                    "prerequisites": {"type": "array", "items": {"type": "string"}},
                    "limitations": {"type": "string"},
                    "practitioner_tldr": {"type": "string"},
                    "tags": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["one_liner", "method_summary", "key_results", "tags"],
            },
        }

        def extract_text(pdf_bytes: bytes) -> str:
            with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
                return "\\n\\n".join(
                    p.extract_text() or "" for p in pdf.pages[:15]
                )[:12000]

        def extract_digest(text: str, title: str, model: str = "claude-sonnet-4-6") -> dict:
            client = _get_client()
            resp = client.messages.create(
                model=model,
                max_tokens=1024,
                tools=[EXTRACT_TOOL],
                tool_choice={"type": "tool", "name": "extract_digest"},
                messages=[{
                    "role": "user",
                    "content": f"Paper title: {title}\\n\\n{text[:10000]}"
                }],
            )
            for block in resp.content:
                if block.type == "tool_use":
                    cost = (
                        resp.usage.input_tokens * 3 / 1_000_000 +
                        resp.usage.output_tokens * 15 / 1_000_000
                    )
                    return {**block.input, "cost_usd": cost, "method": "native"}
            raise RuntimeError("No tool_use block in response")
    """))

# ---------- codegen.py --------------------------------------------------------
with open(f"{BASE}/codegen.py", "w") as f:
    f.write(textwrap.dedent("""
        import os, re, anthropic

        def generate_code_example(one_liner: str, method_summary: str) -> str:
            client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
            resp = client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=512,
                messages=[{
                    "role": "user",
                    "content": (
                        f"Write a minimal Python code snippet (≤40 lines) that demonstrates "
                        f"the core idea of this paper.\\n"
                        f"One-liner: {one_liner}\\n"
                        f"Method: {method_summary}\\n"
                        "Only code, no explanation."
                    ),
                }],
            )
            code = resp.content[0].text
            # Strip markdown fences
            code = re.sub(r'^```[a-z]*\\n?', '', code, flags=re.M).strip('`').strip()
            return code
    """))

# ---------- pipeline.py -------------------------------------------------------
with open(f"{BASE}/pipeline.py", "w") as f:
    f.write(textwrap.dedent("""
        import asyncio, time, json
        from dataclasses import dataclass, field
        from typing import List, Optional
        from .fetch import parse_arxiv_id, fetch_metadata, fetch_pdf_bytes
        from .extract import extract_text, extract_digest
        from .codegen import generate_code_example
        from .models import PaperDigest

        def distill(
            arxiv_id_raw: str,
            model: str = "claude-sonnet-4-6",
            include_code: bool = True,
        ) -> PaperDigest:
            arxiv_id = parse_arxiv_id(arxiv_id_raw)
            meta = fetch_metadata(arxiv_id)
            pdf_bytes = fetch_pdf_bytes(arxiv_id)
            text = extract_text(pdf_bytes)
            fields = extract_digest(text, meta["title"], model=model)
            code = ""
            if include_code:
                try:
                    code = generate_code_example(fields["one_liner"], fields["method_summary"])
                except Exception:
                    code = "# Code generation failed"
            return PaperDigest(
                arxiv_id=arxiv_id,
                title=meta["title"],
                code_example=code,
                **{k: v for k, v in fields.items() if k not in ("cost_usd", "method")},
                cost_usd=fields.get("cost_usd", 0.0),
                method=fields.get("method", "native"),
            )

        @dataclass
        class BatchResult:
            successes: List[PaperDigest] = field(default_factory=list)
            failures: List[dict] = field(default_factory=list)

            @property
            def total_cost(self) -> float:
                return sum(d.cost_usd for d in self.successes)

        def batch_distill(
            arxiv_ids: List[str],
            model: str = "claude-sonnet-4-6",
            include_code: bool = False,
            concurrency: int = 2,
        ) -> BatchResult:
            result = BatchResult()
            for arxiv_id in arxiv_ids:
                try:
                    digest = distill(arxiv_id, model=model, include_code=include_code)
                    result.successes.append(digest)
                except Exception as e:
                    result.failures.append({"arxiv_id": arxiv_id, "error": str(e)})
            return result
    """))

# ---------- evals/golden_harness.py ------------------------------------------
with open(f"{BASE}/evals/__init__.py", "w") as f:
    f.write("")

with open(f"{BASE}/evals/golden_harness.py", "w") as f:
    f.write(textwrap.dedent("""
        from dataclasses import dataclass
        from typing import List

        GOLDEN_SET = [
            {"arxiv_id": "1706.03762", "expected_tags": ["attention", "transformer"], "must_mention": ["attention"]},
            {"arxiv_id": "1810.04805", "expected_tags": ["bert", "pretraining"], "must_mention": ["BERT"]},
            {"arxiv_id": "2106.09685", "expected_tags": ["lora", "fine-tuning"], "must_mention": ["LoRA"]},
        ]

        @dataclass
        class EvalResult:
            arxiv_id: str
            overall_score: float
            pass_: bool

        def run_golden_eval(threshold: float = 0.75) -> List[EvalResult]:
            from paper_distiller.pipeline import distill
            results = []
            for item in GOLDEN_SET:
                try:
                    digest = distill(item["arxiv_id"], include_code=False)
                    text = (digest.one_liner + digest.method_summary + digest.key_results).lower()
                    mentions = sum(1 for m in item["must_mention"] if m.lower() in text)
                    score = mentions / max(len(item["must_mention"]), 1)
                    results.append(EvalResult(arxiv_id=item["arxiv_id"], overall_score=score, pass_=score >= threshold))
                except Exception as e:
                    results.append(EvalResult(arxiv_id=item["arxiv_id"], overall_score=0.0, pass_=False))
            return results
    """))

# ---------- __init__.py -------------------------------------------------------
with open(f"{BASE}/__init__.py", "w") as f:
    f.write(textwrap.dedent("""
        """paper-distiller — turn arXiv papers into practitioner digests."""

        __version__ = "0.3.0"
        __author__ = "Gourav Khanijoe"
        __license__ = "MIT"

        from .models import PaperDigest
        from .pipeline import distill, batch_distill

        __all__ = ["PaperDigest", "distill", "batch_distill", "__version__"]
    """))

print("✅ Module stubs written to /content/paper_distiller/")

# Add to Python path so we can import it
import sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import paper_distiller
print(f"paper_distiller v{paper_distiller.__version__} imported successfully")

## 3. Building the CLI — `cli.py`

A production-grade CLI has four concerns:

1. **Command structure** — top-level `app` with subcommands (`distill`, `batch`, `eval`, `version`)
2. **Rich output** — progress spinners, panels, syntax-highlighted JSON
3. **Error handling** — user-friendly messages, not Python tracebacks
4. **Exit codes** — `0` = success, non-zero = failure (critical for CI pipelines)

### The `typer.Option` patterns you'll use most:

```python
# Short + long flag with env var binding
model: str = typer.Option(
    "claude-sonnet-4-6",
    "--model", "-m",
    help="Claude model to use",
    envvar="PAPER_DISTILLER_MODEL",  # override with env var
)

# Required option (no default)
api_key: str = typer.Option(..., envvar="ANTHROPIC_API_KEY", help="Anthropic API key")

# Path that must exist
input_file: Path = typer.Option(..., exists=True, help="Input file of arXiv IDs")

# Callback for --version flag
def version_callback(value: bool):
    if value:
        typer.echo(f"paper-distiller v{__version__}")
        raise typer.Exit()
```

In [ ]:
# ── CELL 4: Build cli.py ───────────────────────────────────────────────────────
cli_source = '''
"""paper-distiller CLI — distill arXiv papers from the terminal."""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path
from typing import Optional

import typer
from rich.console import Console
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn
from rich.syntax import Syntax
from rich.table import Table
from rich import box

from paper_distiller import __version__, distill, batch_distill
from paper_distiller.models import PaperDigest

# ── App setup ──────────────────────────────────────────────────────────────────
app = typer.Typer(
    name="paper-distiller",
    help="Turn arXiv papers into practitioner digests.",
    rich_markup_mode="rich",   # enables [bold]/[green] in help text
    no_args_is_help=True,      # show help if called with no args
)
console = Console()
err_console = Console(stderr=True, style="bold red")


# ── --version callback ─────────────────────────────────────────────────────────
def _version_callback(value: bool) -> None:
    if value:
        typer.echo(f"paper-distiller v{__version__}")
        raise typer.Exit()


@app.callback()
def main(
    version: Optional[bool] = typer.Option(
        None, "--version", "-v",
        callback=_version_callback,
        is_eager=True,     # evaluated before any subcommand
        help="Show version and exit.",
    ),
) -> None:
    """[bold green]paper-distiller[/bold green] — arXiv → practitioner digest."""


# ── Helper: render digest to terminal ─────────────────────────────────────────
def _render_digest(digest: PaperDigest, fmt: str = "rich") -> None:
    if fmt == "json":
        print(digest.model_dump_json(indent=2))
        return

    if fmt == "markdown":
        lines = [
            f"# {digest.title}",
            f"",
            f"**arXiv:** {digest.arxiv_id}  ",
            f"**Tags:** {', '.join(digest.tags)}",
            f"",
            f"## One-liner",
            digest.one_liner,
            f"",
            f"## Method",
            digest.method_summary,
            f"",
            f"## Key Results",
            digest.key_results,
            f"",
            f"## TL;DR for Practitioners",
            digest.practitioner_tldr,
        ]
        if digest.code_example:
            lines += [f"", f"## Code Example", f"```python", digest.code_example, f"```"]
        print("\n".join(lines))
        return

    # Rich rendering (default)
    console.print(Panel(
        f"[bold cyan]{digest.title}[/bold cyan]\n"
        f"[dim]arxiv:{digest.arxiv_id}  tags: {', '.join(digest.tags)}  "
        f"cost: ${digest.cost_usd:.4f}[/dim]",
        title="📄 Paper Digest",
        expand=False,
    ))

    table = Table(show_header=False, box=box.SIMPLE, padding=(0, 1))
    table.add_column("Field", style="bold yellow", width=20)
    table.add_column("Value")

    rows = [
        ("One-liner", digest.one_liner),
        ("Method", digest.method_summary[:300] + "..." if len(digest.method_summary) > 300 else digest.method_summary),
        ("Key Results", digest.key_results[:300] + "..." if len(digest.key_results) > 300 else digest.key_results),
        ("TL;DR", digest.practitioner_tldr),
        ("Limitations", digest.limitations or "—"),
        ("Prerequisites", ", ".join(digest.prerequisites) or "—"),
    ]
    for field, value in rows:
        table.add_row(field, value)

    console.print(table)

    if digest.code_example:
        console.print(Panel(
            Syntax(digest.code_example, "python", theme="monokai", line_numbers=True),
            title="💻 Code Example",
            expand=False,
        ))


# ── Command: distill ──────────────────────────────────────────────────────────
@app.command("distill")
def cmd_distill(
    arxiv_id: str = typer.Argument(..., help="arXiv ID, URL, or abs path"),
    model: str = typer.Option(
        "claude-sonnet-4-6", "--model", "-m",
        help="Claude model for extraction.",
        envvar="PAPER_DISTILLER_MODEL",
    ),
    output: Optional[Path] = typer.Option(
        None, "--output", "-o",
        help="Write JSON digest to this file.",
    ),
    fmt: str = typer.Option("rich", "--format", "-f", help="Output format: rich | json | markdown"),
    no_code: bool = typer.Option(False, "--no-code", help="Skip code example generation."),
) -> None:
    """Distill a single arXiv paper into a practitioner digest."""
    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        transient=True,
        console=console,
    ) as progress:
        progress.add_task(f"Distilling {arxiv_id}...", total=None)
        try:
            digest = distill(arxiv_id, model=model, include_code=not no_code)
        except ValueError as e:
            err_console.print(f"Invalid arXiv ID: {e}")
            raise typer.Exit(code=1)
        except Exception as e:
            err_console.print(f"Distillation failed: {e}")
            raise typer.Exit(code=2)

    _render_digest(digest, fmt)

    if output:
        output.write_text(digest.model_dump_json(indent=2))
        console.print(f"[dim]Saved to {output}[/dim]")


# ── Command: batch ────────────────────────────────────────────────────────────
@app.command("batch")
def cmd_batch(
    input_file: Path = typer.Argument(
        ...,
        help="File with one arXiv ID per line.",
        exists=True, file_okay=True, readable=True,
    ),
    output: Path = typer.Option(
        Path("digests.jsonl"), "--output", "-o",
        help="Output JSONL file.",
    ),
    model: str = typer.Option(
        "claude-sonnet-4-6", "--model", "-m",
        envvar="PAPER_DISTILLER_MODEL",
    ),
    concurrency: int = typer.Option(2, "--concurrency", "-c", help="Max parallel distillations."),
    no_code: bool = typer.Option(True, "--no-code/--with-code", help="Skip code generation (faster)."),
) -> None:
    """Batch-distill papers from a file of arXiv IDs (one per line)."""
    arxiv_ids = [
        line.strip() for line in input_file.read_text().splitlines()
        if line.strip() and not line.startswith("#")
    ]

    if not arxiv_ids:
        err_console.print("Input file is empty.")
        raise typer.Exit(code=1)

    console.print(f"[bold]Distilling {len(arxiv_ids)} papers...[/bold]")

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        console=console,
    ) as progress:
        task = progress.add_task("Processing...", total=len(arxiv_ids))
        result = batch_distill(arxiv_ids, model=model, include_code=not no_code, concurrency=concurrency)
        progress.update(task, advance=len(arxiv_ids))

    # Write JSONL
    with output.open("w") as f:
        for digest in result.successes:
            f.write(digest.model_dump_json() + "\n")
        for fail in result.failures:
            f.write(json.dumps({"error": True, **fail}) + "\n")

    # Summary table
    table = Table("Metric", "Value", box=box.SIMPLE)
    table.add_row("Papers", str(len(arxiv_ids)))
    table.add_row("Successes", f"[green]{len(result.successes)}[/green]")
    table.add_row("Failures", f"[red]{len(result.failures)}[/red]")
    table.add_row("Total cost", f"${result.total_cost:.4f}")
    table.add_row("Output", str(output))
    console.print(table)

    if result.failures:
        console.print("[yellow]Failed papers:[/yellow]")
        for fail in result.failures:
            console.print(f"  • {fail['arxiv_id']}: {fail['error']}")

    raise typer.Exit(code=0 if not result.failures else 1)


# ── Command: eval ─────────────────────────────────────────────────────────────
@app.command("eval")
def cmd_eval(
    threshold: float = typer.Option(0.75, "--threshold", "-t", help="Minimum score to pass."),
    model: str = typer.Option(
        "claude-sonnet-4-6", "--model", "-m",
        envvar="PAPER_DISTILLER_MODEL",
    ),
) -> None:
    """Run the golden eval harness and gate on threshold (exit 1 if failing)."""
    from paper_distiller.evals.golden_harness import run_golden_eval

    console.print(f"[bold]Running golden eval (threshold={threshold})...[/bold]")

    with Progress(SpinnerColumn(), TextColumn("{task.description}"), transient=True) as p:
        p.add_task("Evaluating golden set...", total=None)
        results = run_golden_eval(threshold=threshold)

    table = Table("arXiv ID", "Score", "Pass", box=box.SIMPLE)
    for r in results:
        pass_str = "[green]✅ PASS[/green]" if r.pass_ else "[red]❌ FAIL[/red]"
        table.add_row(r.arxiv_id, f"{r.overall_score:.2f}", pass_str)
    console.print(table)

    mean_score = sum(r.overall_score for r in results) / len(results) if results else 0
    all_pass = all(r.pass_ for r in results)

    console.print(f"Mean score: {mean_score:.2f} | Gate: {threshold}")

    if not all_pass:
        err_console.print(f"Eval FAILED: mean score {mean_score:.2f} < {threshold}")
        raise typer.Exit(code=1)

    console.print("[green bold]Eval PASSED ✅[/green bold]")


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app()
'''

with open("/content/paper_distiller/cli.py", "w") as f:
    f.write(cli_source)

print("✅ cli.py written")

## 4. Running the CLI in Colab

In production you'd install the package and call `paper-distiller distill 1706.03762` from a terminal.

In Colab we call the command functions directly (same code paths, just no subprocess overhead):

In [ ]:
# ── CELL 5: Demo — distill a single paper ─────────────────────────────────────
# This calls the same function that 'paper-distiller distill 1706.03762' would call
from paper_distiller.cli import cmd_distill

# Equivalent to: paper-distiller distill 1706.03762 --format rich
cmd_distill(
    arxiv_id="1706.03762",
    model="claude-sonnet-4-6",
    output=None,
    fmt="rich",
    no_code=True,         # faster for demo
)

# 💡 EXPERIMENT:
# Change fmt="json" to get machine-readable output
# Change fmt="markdown" to get copy-pasteable Markdown
# Change no_code=False to also generate a code snippet

In [ ]:
# ── CELL 6: Demo — batch distill ──────────────────────────────────────────────
from pathlib import Path

# Write a sample input file
input_file = Path("/content/sample_ids.txt")
input_file.write_text("""# Landmark transformer papers
1706.03762
1810.04805
""")

from paper_distiller.cli import cmd_batch

# Equivalent to:
# paper-distiller batch sample_ids.txt --output digests.jsonl --no-code
cmd_batch(
    input_file=input_file,
    output=Path("/content/digests.jsonl"),
    model="claude-sonnet-4-6",
    concurrency=2,
    no_code=True,
)

# Verify output
print("\n--- First line of JSONL output ---")
import json
with open("/content/digests.jsonl") as f:
    first = json.loads(f.readline())
    print(f"Title: {first.get('title', 'N/A')}")
    print(f"One-liner: {first.get('one_liner', 'N/A')[:100]}...")

## 5. `pyproject.toml` — the modern Python package manifest

`pyproject.toml` replaced `setup.py` + `setup.cfg` + `MANIFEST.in`. It's the single source of truth for:
- Project metadata (name, version, author, license, classifiers)
- Dependencies
- **Entry points** — this is what makes `paper-distiller` a shell command
- Optional dependency groups (`[multimodal]`, `[dev]`, `[all]`)
- Tool configs (ruff, mypy, pytest)

### Entry points explained:
```toml
[project.scripts]
paper-distiller = "paper_distiller.cli:app"
```
When pip installs the package, it creates a `paper-distiller` executable in `~/.local/bin/` that runs `paper_distiller.cli.app()`. That's the entire magic.

In [ ]:
# ── CELL 7: Write pyproject.toml ──────────────────────────────────────────────
import os
PROJ = "/content/paper_distiller_pkg"
os.makedirs(f"{PROJ}/paper_distiller", exist_ok=True)

# Copy our package into the project directory
import shutil
if os.path.exists(f"{PROJ}/paper_distiller"):
    shutil.rmtree(f"{PROJ}/paper_distiller")
shutil.copytree("/content/paper_distiller", f"{PROJ}/paper_distiller")

pyproject = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "paper-distiller"
version = "0.3.0"
description = "Turn arXiv papers into practitioner digests using Claude"
readme = "README.md"
license = { text = "MIT" }
authors = [
  { name = "Gourav Khanijoe", email = "gouravkhanijoe@gmail.com" }
]
requires-python = ">= 3.10"
classifiers = [
  "Development Status :: 3 - Alpha",
  "Intended Audience :: Developers",
  "Intended Audience :: Science/Research",
  "License :: OSI Approved :: MIT License",
  "Programming Language :: Python :: 3",
  "Programming Language :: Python :: 3.10",
  "Programming Language :: Python :: 3.11",
  "Programming Language :: Python :: 3.12",
  "Topic :: Scientific/Engineering :: Artificial Intelligence",
]

# Minimum runtime dependencies
dependencies = [
  "anthropic>=0.30",
  "pydantic>=2.0",
  "requests>=2.28",
  "pdfplumber>=0.10",
  "rich>=13.0",
  "typer[all]>=0.12",
]

# Optional extras — install with pip install paper-distiller[scanned]
[project.optional-dependencies]
scanned = [
  "pdf2image>=1.16",
  "Pillow>=10.0",
]
dev = [
  "pytest>=8.0",
  "pytest-asyncio>=0.23",
  "ruff>=0.4",
  "mypy>=1.8",
  "build>=1.0",
  "twine>=5.0",
]
all = ["paper-distiller[scanned,dev]"]

# ── THIS IS THE MAGIC ─────────────────────────────────────────────────────────
# pip creates a 'paper-distiller' executable pointing to this function
[project.scripts]
paper-distiller = "paper_distiller.cli:app"

[project.urls]
Homepage = "https://github.com/gouravkhanijoe/paper-distiller"
Repository = "https://github.com/gouravkhanijoe/paper-distiller"
Issues = "https://github.com/gouravkhanijoe/paper-distiller/issues"
Changelog = "https://github.com/gouravkhanijoe/paper-distiller/blob/main/CHANGELOG.md"

# ── Tool configs ──────────────────────────────────────────────────────────────
[tool.ruff]
target-version = "py310"
line-length = 100
select = ["E", "F", "I", "UP", "B", "SIM"]

[tool.mypy]
python_version = "3.10"
strict = true
ignore_missing_imports = true

[tool.pytest.ini_options]
testpaths = ["tests"]
asyncio_mode = "auto"

[tool.hatch.build.targets.wheel]
packages = ["paper_distiller"]
'''

with open(f"{PROJ}/pyproject.toml", "w") as f:
    f.write(pyproject)

# README placeholder
readme = """# paper-distiller

Turn arXiv papers into practitioner digests using Claude.

```bash
pip install paper-distiller
paper-distiller distill 1706.03762
```
"""
with open(f"{PROJ}/README.md", "w") as f:
    f.write(readme)

# py.typed marker — tells mypy this package has type annotations
open(f"{PROJ}/paper_distiller/py.typed", "w").close()

print("✅ pyproject.toml written")
print("\nProject structure:")
!find {PROJ} -type f | sort

## 6. Building the package

```
python -m build
```
produces two artifacts in `dist/`:

| Artifact | What it is | When used |
|----------|-----------|----------|
| `paper_distiller-0.3.0-py3-none-any.whl` | **Wheel** — pre-built, fast to install | PyPI install |
| `paper-distiller-0.3.0.tar.gz` | **Source dist (sdist)** — full source | Fallback / build-from-source |

`twine check dist/*` verifies both artifacts are well-formed before uploading.

The `py3-none-any` in the wheel name means:
- `py3` — Python 3 only
- `none` — no C extensions (pure Python)
- `any` — works on any OS/arch

In [ ]:
# ── CELL 8: Build the wheel ────────────────────────────────────────────────────
!pip install hatchling -q

import subprocess, os

result = subprocess.run(
    ["python", "-m", "build", "/content/paper_distiller_pkg"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
else:
    print("✅ Build succeeded")

print("\nArtifacts produced:")
!ls -lh /content/paper_distiller_pkg/dist/ 2>/dev/null || echo "dist/ not found — check build output above"

# Verify the wheel is well-formed
!twine check /content/paper_distiller_pkg/dist/* 2>/dev/null || echo "twine check needs dist/*.whl"

## 7. Installing from the wheel (local test)

Before uploading to PyPI, always test installing your own wheel:

```bash
pip install dist/paper_distiller-0.3.0-py3-none-any.whl
paper-distiller --version  # should print 0.3.0
paper-distiller --help     # should show all commands
```

In [ ]:
# ── CELL 9: Install from local wheel and test ──────────────────────────────────
import glob

wheels = glob.glob("/content/paper_distiller_pkg/dist/*.whl")
if wheels:
    wheel = wheels[0]
    print(f"Installing: {wheel}")
    !pip install "{wheel}" --force-reinstall -q
    print("\n--- paper-distiller --version ---")
    !paper-distiller --version
    print("\n--- paper-distiller --help ---")
    !paper-distiller --help
else:
    print("No wheel found. Run cell 8 first.")

## 8. Publishing to PyPI

### Step 1: TestPyPI first (always)

TestPyPI (`test.pypi.org`) is a staging environment — upload there first, install from there, verify it works, then publish to real PyPI.

```bash
# Upload to TestPyPI
twine upload --repository testpypi dist/*

# Install from TestPyPI (note the extra-index-url for dependencies)
pip install --index-url https://test.pypi.org/simple/ \
            --extra-index-url https://pypi.org/simple/ \
            paper-distiller
```

### Step 2: Real PyPI

```bash
twine upload dist/*
```

### Step 3: OIDC Trusted Publishing (no tokens in CI)

Instead of storing a `PYPI_API_TOKEN` secret in GitHub Actions, PyPI supports OIDC-based trust:

1. Go to your PyPI project → **Publishing** tab
2. Add a trusted publisher: GitHub org `gouravkhanijoe`, repo `paper-distiller`, workflow `release.yml`
3. In your workflow, add `id-token: write` permission and use the official publish action

No token ever stored anywhere — PyPI verifies the GitHub Actions OIDC token directly.

In [ ]:
# ── CELL 10: GitHub Actions — CI + release workflow ────────────────────────────
import os
os.makedirs("/content/paper_distiller_pkg/.github/workflows", exist_ok=True)

# ---- ci.yml (runs on every push/PR) -----------------------------------------
ci_yml = """
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install dev dependencies
        run: pip install -e .[dev]

      - name: Lint (ruff)
        run: ruff check paper_distiller/

      - name: Type check (mypy)
        run: mypy paper_distiller/

      - name: Unit tests
        run: pytest tests/ -x -q

  build:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install build twine
      - run: python -m build
      - run: twine check dist/*
"""

# ---- release.yml (runs only on pushed tags like v0.3.0) ----------------------
release_yml = """
name: Release to PyPI

on:
  push:
    tags:
      - "v*"    # triggers on v0.3.0, v1.0.0, etc.

permissions:
  id-token: write   # OIDC Trusted Publishing — no API token needed
  contents: read

jobs:
  release:
    runs-on: ubuntu-latest
    environment: pypi   # requires 'pypi' environment to be configured in GitHub

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }

      - name: Build
        run: |
          pip install build twine
          python -m build
          twine check dist/*

      # ✅ OIDC publish — no PYPI_API_TOKEN secret needed
      - name: Publish to PyPI
        uses: pypa/gh-action-pypi-publish@release/v1
        # No 'with: password:' needed — OIDC handles auth
"""

with open("/content/paper_distiller_pkg/.github/workflows/ci.yml", "w") as f:
    f.write(ci_yml)

with open("/content/paper_distiller_pkg/.github/workflows/release.yml", "w") as f:
    f.write(release_yml)

print("✅ GitHub Actions workflows written")

# Print step-by-step release instructions
print("""
===== Release checklist =====

1. Bump version in pyproject.toml AND paper_distiller/__init__.py
2. Update CHANGELOG.md
3. git add . && git commit -m "chore: release v0.3.0"
4. git tag v0.3.0
5. git push && git push --tags
   → release.yml triggers automatically
   → OIDC token minted, wheel built, uploaded to PyPI
6. gh release create v0.3.0 --generate-notes

One-time PyPI setup (do this once per project):
  1. Create account at pypi.org
  2. Project → Publishing → Add trusted publisher
     Owner: gouravkhanijoe
     Repo: paper-distiller
     Workflow: release.yml
  3. Done. No tokens to rotate.
""")

## 9. Shell completion

Typer can generate shell completion scripts for `bash`, `zsh`, and `fish`. This lets users press `Tab` to autocomplete commands and flags.

```bash
# Generate and install completions (one-time setup)

# bash
paper-distiller --install-completion bash

# zsh
paper-distiller --install-completion zsh

# fish
paper-distiller --install-completion fish
```

After running the install command, new terminals will have tab completion for all `paper-distiller` subcommands and flags.

This comes **for free** with `typer[all]` — no extra code required.

In [ ]:
# ── CELL 11: Shell completion demo ────────────────────────────────────────────
# Show the completion script (what gets installed to ~/.bashrc)
from paper_distiller.cli import app as pd_app
from typer.testing import CliRunner

runner = CliRunner()

# Test --help for each subcommand
for cmd in ["--help", "distill --help", "batch --help", "eval --help"]:
    result = runner.invoke(pd_app, cmd.split())
    print(f"\n{'='*60}")
    print(f"$ paper-distiller {cmd}")
    print('='*60)
    print(result.output[:800])  # truncate for display

# 💡 EXPERIMENT:
# result = runner.invoke(pd_app, ["distill", "invalid-id"])
# print(result.output)  # should show an error message
# print("Exit code:", result.exit_code)  # should be non-zero

## 10. Ten pitfalls when building CLIs and packaging for PyPI

| # | Pitfall | Fix |
|---|---------|-----|
| 1 | **`sys.exit()` vs `raise typer.Exit(code=1)`** — using `sys.exit()` in a Typer command bypasses Typer's cleanup and breaks the test runner | Always use `raise typer.Exit(code=N)` inside Typer commands |
| 2 | **Printing to stdout when CI pipes to a file** — Rich detects `stdout.isatty()` and strips colors automatically, but you might accidentally print structured data mixed with human text | Use `--format json` pattern; never mix JSON and prose in the same stdout stream |
| 3 | **No `py.typed` marker** — without an empty `py.typed` file, mypy ignores your package's type annotations in downstream projects | Add `py.typed` to `paper_distiller/` and include it in the wheel via `hatch.build.targets.wheel` |
| 4 | **Version mismatch between `__init__.py` and `pyproject.toml`** — a common copy-paste error | Use a single source of truth: read `__version__` from `pyproject.toml` at import time via `importlib.metadata.version('paper-distiller')` |
| 5 | **Publishing without TestPyPI first** — once uploaded to PyPI, you can't delete a version (only yank it) | Always TestPyPI → local pip install → real PyPI |
| 6 | **`is_eager=True` missing on `--version`** — without it, Typer tries to validate positional args before running the version callback, so `paper-distiller --version` crashes | Set `is_eager=True` on all meta-callbacks (`--version`, `--verbose`) |
| 7 | **Progress bars on non-TTY outputs** — progress bars written to a pipe (like `paper-distiller distill 1706.03762 > out.json`) pollute the output | Use `transient=True` and/or redirect progress to `stderr` not `stdout` |
| 8 | **Catching too broadly in CLI commands** — `except Exception` hides bugs during development | Catch specific exceptions (`ValueError`, `requests.HTTPError`) and let unexpected exceptions propagate in debug mode |
| 9 | **Missing `[project.scripts]` in pyproject.toml** — the package installs but `paper-distiller` isn't on PATH | Double-check `[project.scripts]` is present and the entry point path is correct (`module.path:function`) |
| 10 | **Large binary files in the sdist** — accidentally including `.whl` or test PDFs inflates the sdist and can block PyPI upload | Add a `.gitignore`-style `[tool.hatch.build.targets.sdist] exclude` list to `pyproject.toml` |

In [ ]:
# ── CELL 12: Pitfall demo — single source of truth for version ─────────────────

# BAD: version duplicated in __init__.py and pyproject.toml → drift over time
# __version__ = "0.3.0"  # in __init__.py
# version = "0.3.0"       # in pyproject.toml

# GOOD: read version from installed package metadata
from importlib.metadata import version, PackageNotFoundError

def get_version() -> str:
    try:
        return version("paper-distiller")
    except PackageNotFoundError:
        # Fallback for development (not installed yet)
        return "0.0.0-dev"

pkg_version = get_version()
print(f"Package version (from metadata): {pkg_version}")

# This is what pyproject.toml says
import tomllib
with open("/content/paper_distiller_pkg/pyproject.toml", "rb") as f:
    config = tomllib.load(f)
pyproject_version = config["project"]["version"]
print(f"pyproject.toml version: {pyproject_version}")

# In __init__.py, use this pattern instead of hardcoding:
print("""
# In paper_distiller/__init__.py — the modern way:
try:
    from importlib.metadata import version
    __version__ = version("paper-distiller")
except ImportError:  # Python < 3.8
    from importlib_metadata import version
    __version__ = version("paper-distiller")
except PackageNotFoundError:
    __version__ = "0.0.0-dev"  # running from source without install
""")

In [ ]:
# ── CELL 13: Verification — full project structure ─────────────────────────────
from rich.console import Console
from rich.tree import Tree

c = Console()

# Print project tree
tree = Tree("📦 paper_distiller_pkg/")

import os
for root, dirs, files in os.walk("/content/paper_distiller_pkg"):
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".eggs", "*.egg-info")]
    level = root.replace("/content/paper_distiller_pkg", "").count(os.sep)
    if level > 3:
        continue
    indent = "  " * level
    folder = os.path.basename(root)
    subtree = tree.add(f"📁 {folder}/") if level > 0 else tree
    for f in sorted(files):
        subtree.add(f"📄 {f}")

c.print(tree)

# Checklist
checklist = [
    ("pyproject.toml", os.path.exists("/content/paper_distiller_pkg/pyproject.toml")),
    ("README.md", os.path.exists("/content/paper_distiller_pkg/README.md")),
    ("paper_distiller/__init__.py", os.path.exists("/content/paper_distiller_pkg/paper_distiller/__init__.py")),
    ("paper_distiller/cli.py", os.path.exists("/content/paper_distiller_pkg/paper_distiller/cli.py")),
    ("paper_distiller/py.typed", os.path.exists("/content/paper_distiller_pkg/paper_distiller/py.typed")),
    (".github/workflows/ci.yml", os.path.exists("/content/paper_distiller_pkg/.github/workflows/ci.yml")),
    (".github/workflows/release.yml", os.path.exists("/content/paper_distiller_pkg/.github/workflows/release.yml")),
    ("dist/*.whl exists", bool(glob.glob("/content/paper_distiller_pkg/dist/*.whl"))),
]

c.print("\n[bold]Checklist:[/bold]")
import glob
for item, ok in checklist:
    status = "[green]✅[/green]" if ok else "[red]❌[/red]"
    c.print(f"  {status} {item}")

print("\nPhase 6 Lesson 3 complete ✅")

## Summary

| Concept | What you learned |
|---------|------------------|
| **Typer** | Function signature = CLI schema; `typer.Argument` vs `typer.Option`; `Enum` for allowed choices |
| **Rich integration** | `Progress` spinners/bars, `Panel`, `Syntax`, `Table` — all zero-cost with Typer's `[all]` extra |
| **Error handling** | Specific `except` clauses + `raise typer.Exit(code=N)` for clean exit codes |
| **`pyproject.toml`** | Single manifest for metadata, deps, entry points, tool configs |
| **Entry points** | `[project.scripts]` = the line that creates the `paper-distiller` shell command |
| **Build** | `python -m build` → wheel + sdist; `twine check` before uploading |
| **OIDC publishing** | Trusted Publisher on PyPI — no API tokens, GitHub Actions OIDC does auth |
| **Shell completion** | Free with `typer[all]`; `--install-completion bash/zsh/fish` |

## Homework

1. **Add a `config` subcommand** — `paper-distiller config set model claude-haiku-4-5` that writes to `~/.paper_distiller.toml` and `paper-distiller config show` that prints the current config. Use `pydantic-settings` with `toml` file support.

2. **Wire `importlib.metadata.version`** — replace the hardcoded `__version__ = "0.3.0"` in `__init__.py` with the metadata approach from Cell 12.

3. **Write a unit test for the CLI** — use `typer.testing.CliRunner` to test that `paper-distiller distill invalid-id` exits with code 1 (no real API calls needed).

4. **Publish to TestPyPI** — create a free account at `test.pypi.org` and do a real `twine upload --repository testpypi dist/*`. Then install it with `pip install --index-url https://test.pypi.org/simple/ paper-distiller`.

5. **Benchmark build time** — add a `build` job to `ci.yml` that caches the pip wheel cache with `actions/cache`. Measure how much faster repeat builds are.

## Next lesson: L59 — FastAPI Web API

We'll expose `paper-distiller` as an HTTP service:
```
POST /distill     { arxiv_id: "1706.03762" }  → PaperDigest
POST /batch       { arxiv_ids: [...] }         → BatchResult (streaming JSONL)
GET  /health      {}                           → { status: "ok", version: "0.3.0" }
GET  /metrics     {}                           → Prometheus text format
```

This turns paper-distiller into a microservice that other agents can call via HTTP — the same pattern you'll use to expose any AI tool in a production system.